In [ ]:
# Cell1：导入包 + 读取清洗好的数据
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
plt.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei']
plt.rcParams['axes.unicode_minus'] = False

columns = ['user_id', 'item_id', 'category_id', 'behavior_type', 'timestamp']
df = pd.read_csv('../data/raw/UserBehavior.csv', names=columns)
df['datetime'] = pd.to_datetime(df['timestamp'], unit='s')
df['date'] = df['datetime'].dt.date
start_date = pd.to_datetime('2017-11-25')
end_date = pd.to_datetime('2017-12-04')
df = df[(df['datetime'] >= start_date) & (df['datetime'] < end_date)]

In [ ]:
# Cell2：只拿购买行为，计算RFM
buy_df = df[df['behavior_type'] == 'buy'].copy()
max_date = buy_df['datetime'].max()

rfm = buy_df.groupby('user_id').agg(
    R = ('datetime', lambda x: (max_date - x.max()).days),
    F = ('datetime', 'count'),
    M = ('datetime', 'count')
).reset_index()

# R越小越好，F/M越大越好
rfm['R_score'] = pd.qcut(rfm['R'], q=3, labels=[3,2,1])
rfm['F_score'] = pd.qcut(rfm['F'].rank(method='first'), q=3, labels=[1,2,3])
rfm['M_score'] = pd.qcut(rfm['M'].rank(method='first'), q=3, labels=[1,2,3])

rfm['RFM_label'] = rfm['R_score'].astype(str)+rfm['F_score'].astype(str)+rfm['M_score'].astype(str)
rfm.head()

In [ ]:
# Cell3：用户分层映射
def segment_label(s):
    if s == '333':
        return '高价值用户'
    elif s[0] == '3' and s[1] in ['2','3']:
        return '潜力用户'
    elif s[0] == '1' and s[1] in ['2','3']:
        return '流失高价值用户'
    elif s[0] == '3':
        return '新用户'
    else:
        return '一般流失用户'

rfm['user_segment'] = rfm['RFM_label'].apply(segment_label)
seg_count = rfm['user_segment'].value_counts()
print(seg_count)

plt.figure(figsize=(8,4))
sns.barplot(x=seg_count.index, y=seg_count.values)
plt.title('RFM用户分层数量分布')
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig('../img/rfm_segment.png', dpi=300)
plt.show()

In [ ]:
# Cell4：导出分层结果，给看板用
rfm.to_csv('../data/processed/rfm_segment_result.csv', index=False)
rfm[['user_id','R','F','M','user_segment']].head()